In [ ]:
import  osiris_utils as ou
from matplotlib import pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FuncFormatter
import matplotlib.colors as colors
from pathlib import Path
import sys

mirror_module_dir = Path("/home/exxxx5/Tese/Decks/StudyConvergence")
if str(mirror_module_dir) not in sys.path:
    sys.path.insert(0, str(mirror_module_dir))

from theoretical_mirror import theoretical_mirror_position

plt.rcParams['font.size'] = 14


In [ ]:
def createSimDic(path, sim_labels, test):
    sim = {}
    for key in sim_labels.keys():
        sim[key] = {}
        for dtw in sim_labels[key]:
            sim[key][dtw] = ou.Simulation(f"{path}/{test}/{key}/dtw{dtw}/{key}.in")
    return sim


def createSimDic_nx(path, sim_labels, test):
    sim = {}
    for key in sim_labels.keys():
        sim[key] = {}
        for dtw in sim_labels[key]:
            sim[key][dtw] = ou.Simulation(f"{path}/{test}/{key}/nx{dtw}/{key}.in")
    return sim


def createRawDic(path, sim_labels, test, *, strict=True):
    rawdic = {}
    missing = []
    for key in sim_labels.keys():
        rawdic[key] = {}
        for dtw in sim_labels[key]:
            raw_dir = Path(path) / test / key / f"dtw{dtw}" / "MS" / "RAW" / "test_electrons"
            raw_files = sorted(raw_dir.glob("RAW-test_electrons-*.h5"))
            if not raw_files:
                missing.append(raw_dir)
                continue
            rawdic[key][dtw] = [ou.OsirisRawFile(raw_file) for raw_file in raw_files]

        if not rawdic[key]:
            del rawdic[key]

    if missing:
        message = "No RAW files found in:\n" + "\n".join(str(raw_dir) for raw_dir in missing)
        if strict:
            raise FileNotFoundError(message)
        print(message)

    return rawdic


def createRawDic_nx(path, sim_labels, test):
    rawdic = {}
    for key in sim_labels.keys():
        rawdic[key] = {}
        for dtw in sim_labels[key]:
            raw_dir = Path(path) / test / key / f"nx{dtw}" / "MS" / "RAW" / "test_electrons"
            raw_files = sorted(raw_dir.glob("RAW-test_electrons-*.h5"))
            if not raw_files:
                raise FileNotFoundError(f"No RAW files found in {raw_dir}")
            rawdic[key][dtw] = [ou.OsirisRawFile(raw_file) for raw_file in raw_files]
    return rawdic


In [ ]:
# Normalize axis to w_ce


def _set_scaled_formatter(axis, scale_factor, fmt=".2f"):
    axis.set_major_formatter(
        FuncFormatter(lambda v, pos: f"{v*scale_factor:{fmt}}")
    )

def scale_x_ax(scale_factor, fig, ax, label=r"$t[1 / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_xlabel(label)
    if lock_ticks:
        # freeze current tick positions
        ticks = ax.get_xticks()
        ax.xaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.xaxis, scale_factor, fmt)
    return fig, ax

def scale_y_ax(scale_factor, fig, ax, label=r"$x[c / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_ylabel(label)
    if lock_ticks:
        ticks = ax.get_yticks()
        ax.yaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.yaxis, scale_factor, fmt)
    return fig, ax

def scale_z_ax(scale_factor, fig, ax, label=r"$x[c / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_zlabel(label)
    if lock_ticks:
        ticks = ax.get_zticks()
        ax.zaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.zaxis, scale_factor, fmt)
    return fig, ax

def scale_3d_axes(scale_factor, fig, ax, fmt=".2f"):
    # ax.set_xlabel(r"$x_1[c / \Omega_e]$")
    # ax.set_ylabel(r"$x_2[c / \Omega_e]$")
    # ax.set_zlabel(r"$x_3[c / \Omega_e]$")
    _set_scaled_formatter(ax.xaxis, scale_factor, fmt)
    _set_scaled_formatter(ax.yaxis, scale_factor, fmt)
    _set_scaled_formatter(ax.zaxis, scale_factor, fmt)
    return fig, ax


In [ ]:
def sort_raw_by_tag(raw):
    tag = raw.data["tag"]
    order = np.lexsort((tag[:, 1], tag[:, 0]))

    for key, arr in raw.data.items():
        if hasattr(arr, "shape") and arr.shape[0] == len(order):
            raw.data[key] = arr[order]

    return order


In [ ]:
class curvDriftTheo:
    def __init__(self, sim, B=None, rqm=-1, direc=-1):
        self.rqm = rqm
        self.direc = direc
        self.sim = sim

        tracks = sim["test_electrons"]["tracks"]

        self.x1_0 = tracks["x1"][:, 0]
        self.x2_0 = tracks["x2"][:, 0]

        # 3D if x3 exists, otherwise 2D
        self.is_3d = "x3" in tracks.quants

        if self.is_3d:
            self.x3_0 = tracks["x3"][:, 0]
        else:
            # Dummy x3 only for formulas that accept x1, x2, x3.
            # It is not returned for 2D simulations.
            self.x3_0 = np.zeros_like(self.x1_0)

        self.phi0 = np.arctan2(self.x2_0, self.x1_0)

        self.p1_0 = tracks["p1"][:, 0]
        self.p2_0 = tracks["p2"][:, 0]
        self.p3_0 = tracks["p3"][:, 0]

        self.gamma_0 = np.sqrt(
            1
            + self.p1_0**2
            + self.p2_0**2
            + self.p3_0**2
        )

        self.R0 = np.sqrt(self.x1_0**2 + self.x2_0**2)

        if B is not None:
            self.B0 = B
        else:
            B1 = tracks["B1"][:, 0]
            B2 = tracks["B2"][:, 0]
            B3 = tracks["B3"][:, 0]

            self.B0 = np.sqrt(B1**2 + B2**2 + B3**2)

        self.vc, self.v_par = self._curv_v()

    def b1(self, x1, x2, x3=None):
        return self.B0 * (-x2) / np.sqrt(x1**2 + x2**2)

    def b2(self, x1, x2, x3=None):
        return self.B0 * x1 / np.sqrt(x1**2 + x2**2)

    def b3(self, x1, x2, x3=None):
        return np.zeros_like(x1)

    def _curv_v(self):
        p_par0 = (
            self.p1_0 * self.b1(self.x1_0, self.x2_0, self.x3_0)
            + self.p2_0 * self.b2(self.x1_0, self.x2_0, self.x3_0)
            + self.p3_0 * self.b3(self.x1_0, self.x2_0, self.x3_0)
        ) / self.B0

        v_par = p_par0 / self.gamma_0

        vc = (
            self.rqm
            * p_par0**2
            / self.B0
            / self.gamma_0
            * self.direc
            / self.R0
        )

        return vc, v_par

    def get_curv_drift_v(self):
        return self._curv_v()[0]
    
    def get_curv_drift_p(self):
        return self._curv_v()[0] * self.gamma_0

    def get_curv_traj(self, t):
        t = np.asarray(t, dtype=float)

        omega = self.v_par / self.R0

        phase = np.outer(omega, t) + self.phi0[:, None]

        x1 = self.R0[:, None] * np.cos(phase)
        x2 = self.R0[:, None] * np.sin(phase)

        if self.is_3d:
            x3 = self.x3_0[:, None] + np.outer(self.vc, t)
            return np.array([x1, x2, x3])

        return np.array([x1, x2])
    



class curvDriftTheo_Raw:
    def __init__(self, raws, B, rqm=-1, direc=-1):
        self.rqm = rqm
        self.direc = direc
        self.raws = raws

        data0 = raws[0].data

        self.x1_0 = data0["x1"]
        self.x2_0 = data0["x2"]

        # 3D if x3 exists, otherwise 2D
        self.is_3d = "x3" in data0.quants

        if self.is_3d:
            self.x3_0 = data0["x3"]
        else:
            # Dummy x3 only for formulas that accept x1, x2, x3.
            # It is not returned for 2D simulations.
            self.x3_0 = np.zeros_like(self.x1_0)

        self.phi0 = np.arctan2(self.x2_0, self.x1_0)

        self.p1_0 = data0["p1"]
        self.p2_0 = data0["p2"]
        self.p3_0 = data0["p3"]

        self.gamma_0 = np.sqrt(
            1
            + self.p1_0**2
            + self.p2_0**2
            + self.p3_0**2
        )

        self.R0 = np.sqrt(self.x1_0**2 + self.x2_0**2)

        self.B0 = B

        self.vc, self.v_par = self._curv_v()

    def b1(self, x1, x2, x3=None):
        return self.B0 * (-x2) / np.sqrt(x1**2 + x2**2)

    def b2(self, x1, x2, x3=None):
        return self.B0 * x1 / np.sqrt(x1**2 + x2**2)

    def b3(self, x1, x2, x3=None):
        return np.zeros_like(x1)

    def _curv_v(self):
        p_par0 = (
            self.p1_0 * self.b1(self.x1_0, self.x2_0, self.x3_0)
            + self.p2_0 * self.b2(self.x1_0, self.x2_0, self.x3_0)
            + self.p3_0 * self.b3(self.x1_0, self.x2_0, self.x3_0)
        ) / self.B0

        v_par = p_par0 / self.gamma_0

        vc = (
            self.rqm
            * p_par0**2
            / self.B0
            / self.gamma_0
            * self.direc
            / self.R0
        )

        return vc, v_par

    def get_curv_traj(self, t):
        t = float(t)

        omega = self.v_par / self.R0

        phase = omega * t + self.phi0

        x1 = self.R0 * np.cos(phase)
        x2 = self.R0 * np.sin(phase)

        if self.is_3d:
            x3 = self.x3_0 + self.vc * t
            return np.array([x1, x2, x3])

        return np.array([x1, x2])

        return np.array([x1, x2, x3])
    

In [ ]:
test = "Curv"
path = f"/home/exxxx5/Tese/Decks/EndProdTests/ConvergenceTests"
sim_labels = {
    # 'Curv_Gca_nx80': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    # 'Curv_Gca_nx120': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    # 'Curv_Gca_nx160': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    # 'Curv_Gca_nx200': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    'Curv_Gca_expl_nx80': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    'Curv_Gca_expl_nx120': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    'Curv_Gca_expl_nx160': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    'Curv_Gca_expl_nx200': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Curv_Gca_expl_nx80"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]
machine_pre = 10**-16

components = ["r", "x3"]
# Edit these labels to control the names shown in the legend.
pusher_plot_names = {
    # "Gca": "OldGca_nx80",
    # "gcaCorrV5": "Gca_expl",
}
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
mach_prec_plotted = [False] * len(axes)
for pusher in sim.keys():
    print(f"Processing {pusher}...")
    X = []    
    Y = []
    YERR = []    
    YMax = []
    mach_prec = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        # num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        num_steps = float(dtw.replace("_", "."))  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))
        mach_prec.append(machine_pre / L / num_steps)

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)
    MachPrec = np.asarray(mach_prec, dtype=float)

    for i, ax in enumerate(axes):
        if not mach_prec_plotted[i]:
            ax.plot(
                X,
                MachPrec,
                linestyle=':',
                linewidth=1.2,
                label="double machine precision",
                color="gray",
            )
            mach_prec_plotted[i] = True

        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=pusher_plot_names.get(pusher, f"{pusher[16:]} cells"),
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error / dt / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("Gca-Explicit pusher, 1 step")
axes[0].legend()
axes[-1].set_xlabel(r"$dt \Omega_e$")
fig.tight_layout()


# Mirror test

In [ ]:
test = "Mirror"
path = f"/home/exxxx5/Tese/Decks/EndProdTests/ConvergenceTests"
sim_labels = {
    # 'Mirror_Gca_nx80': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    # 'Mirror_Gca_nx120': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    # 'Mirror_Gca_nx160': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    # 'Mirror_Gca_nx200': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    'Mirror_Gca_expl_nx80': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    'Mirror_Gca_expl_nx120': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    'Mirror_Gca_expl_nx160': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    'Mirror_Gca_expl_nx200': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
}


sim = createSimDic(path, sim_labels, test)


In [ ]:


MIRROR_A = 0.1
MIRROR_B0 = 1000.0
MIRROR_C = 1.0
MIRROR_REFERENCE_N_GRID = 5000
MIRROR_N_PARTICLES = 2300
machine_pre = 10**-16

def first_mirror_particles(tracks, n_particles=MIRROR_N_PARTICLES):
    n_available = min(tracks[name].shape[0] for name in ("x1", "x2", "x3", "p1", "p2", "p3", "t"))
    if n_available < n_particles:
        raise ValueError(f"Requested {n_particles} particles, but tracks only contains {n_available}.")
    return slice(0, n_particles)


def mirror_reference_positions(tracks, *, n_particles=MIRROR_N_PARTICLES, n_grid=MIRROR_REFERENCE_N_GRID):
    particle_sel = first_mirror_particles(tracks, n_particles)
    R0 = np.column_stack((tracks["x1"][particle_sel, 0], tracks["x2"][particle_sel, 0], tracks["x3"][particle_sel, 0]))
    u0 = np.column_stack((tracks["p1"][particle_sel, 0], tracks["p2"][particle_sel, 0], tracks["p3"][particle_sel, 0]))
    dt = np.asarray(tracks["t"][particle_sel, 1] - tracks["t"][particle_sel, 0], dtype=float)

    pos_ref = np.empty_like(R0)
    for i, (R0_i, u0_i, dt_i) in enumerate(zip(R0, u0, dt)):
        # Use quadrature for the smallest physical steps, where grid interpolation
        # error can dominate the dtw-normalized result.
        if dt_i <= 1e-5:
            method = "quad"
            grid_size = 256
        else:
            method = "grid"
            grid_size = n_grid

        pos_ref[i] = theoretical_mirror_position(
            dt_i,
            R0_i,
            u0_i,
            a=MIRROR_A,
            B0=MIRROR_B0,
            c=MIRROR_C,
            n_grid=grid_size,
            rtol=1e-12,
            atol=1e-14,
            method=method,
        )

    return pos_ref, dt


grid = sim["Mirror_Gca_expl_nx80"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "z"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
mach_prec_plotted = [False] * len(axes)

for pusher in sim.keys():
    print(f"Processing {pusher}...")
    X = []
    Y = []
    YERR = []
    YMax = []
    mach_prec = []

    for dtw in sim[pusher].keys():
        num_steps = float(dtw.replace("_", "."))
        tracks = sim[pusher][dtw]["test_electrons"]["tracks"]
        particle_sel = first_mirror_particles(tracks)
        pos_ref, dt = mirror_reference_positions(tracks)

        r_ref = np.sqrt(pos_ref[:, 0]**2 + pos_ref[:, 1]**2)
        z_ref = pos_ref[:, 2]

        r_sim = np.sqrt(tracks["x1"][particle_sel, 1]**2 + tracks["x2"][particle_sel, 1]**2)
        z_sim = tracks["x3"][particle_sel, 1]

        err = np.array([
            np.abs(r_sim - r_ref) / L / num_steps,
            np.abs(z_sim - z_ref) / L / num_steps,
        ])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))
        mach_prec.append(machine_pre / L / num_steps)

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)
    MachPrec = np.asarray(mach_prec, dtype=float)

    order = np.argsort(X)
    X = X[order]
    Y = Y[order]
    YERR = YERR[order]
    YMax = YMax[order]
    MachPrec = MachPrec[order]

    for i, ax in enumerate(axes):
        if not mach_prec_plotted[i]:
            ax.plot(
                X,
                MachPrec,
                linestyle=':',
                linewidth=1.2,
                label="double machine precision",
                color="gray",
            )
            mach_prec_plotted[i] = True

        eb = ax.errorbar(
            X,
            Y[:, i],
            yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle='-',
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error / dtw / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("Gca-Explicit pusher, 1st step")
axes[0].legend()
axes[-1].set_xlabel(r"$dt \Omega_e$")
fig.tight_layout()
